Notebook 1 - RAG-Based Customer Support Bot

Overview:
This notebook implements a Retrieval-Augmented Generation (RAG) approach
for customer support comparison against an Agentic bot.

How it works:
1. Load the Bitext customer support dataset from HuggingFace
2. Split 80% as the FAISS knowledge base, hold out 20% as a clean benchmark
   (this separation prevents evaluation bias - the judge never sees RAG's source data)
3. Embed responses using sentence-transformers (all-MiniLM-L6-v2, CPU)
4. Store embeddings in a FAISS index for fast similarity search
5. At query time: embed query -> retrieve top-3 chunks -> pass to Groq Llama 3.1
6. Save results including retrieved chunk text (needed for hallucination checks in NB3)
7. Compute BLEU + ROUGE against a shared policy-based reference (fair cross-bot metric)
8. Launch Gradio chat UI



In [9]:
# STEP 0 - Install dependencies

!pip install datasets langchain langchain-community faiss-cpu \
             sentence-transformers groq gradio tqdm \
             rouge-score sacrebleu -q

In [10]:
# STEP 1 - Configuration

import os
import json
import numpy as np
import pandas as pd
import time
import random
from google.colab import userdata

api_key = userdata.get('CST_API')

BENCHMARK_CSV   = "benchmark_queries.csv"
RAG_RESULTS_CSV = "rag_results.csv"

EMBED_MODEL  = "all-MiniLM-L6-v2"
GROQ_MODEL   = "llama-3.1-8b-instant"
BATCH_SIZE   = 512
TOP_K        = 3
TRAIN_RATIO  = 0.80   # 80% goes into FAISS index, 20% is held out for benchmark
RANDOM_SEED  = 42

print("Config loaded.")
print(f"   Embedding model : {EMBED_MODEL}")
print(f"   LLM             : {GROQ_MODEL}")
print(f"   Train/test split: {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)}")
print(f"   Top-K retrieval : {TOP_K}")

Config loaded.
   Embedding model : all-MiniLM-L6-v2
   LLM             : llama-3.1-8b-instant
   Train/test split: 80/19
   Top-K retrieval : 3


In [8]:
# STEP 2 - Load and Split Bitext Dataset
# The 80/20 split is critical: RAG retrieves from the training split only.
# The benchmark (held-out 20%) is never seen by the FAISS index,
# so retrieval cannot trivially match ground truth by distribution.

from datasets import load_dataset

print("Loading Bitext dataset...")
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df_full = ds["train"].to_pandas()

print(f"Dataset loaded: {len(df_full):,} rows, {df_full['intent'].nunique()} intents")

# Stratified split by intent so all intents appear in both halves
from sklearn.model_selection import train_test_split

df_train, df_holdout = train_test_split(
    df_full,
    test_size=(1 - TRAIN_RATIO),
    stratify=df_full["intent"],
    random_state=RANDOM_SEED
)

df_train   = df_train.reset_index(drop=True)
df_holdout = df_holdout.reset_index(drop=True)

print(f"Training (FAISS) split : {len(df_train):,} rows")
print(f"Holdout (benchmark)    : {len(df_holdout):,} rows")

Loading Bitext dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset loaded: 26,872 rows, 27 intents
Training (FAISS) split : 21,497 rows
Holdout (benchmark)    : 5,375 rows


In [11]:
# STEP 3 - Build Benchmark from HELD-OUT Split
# Sampling only from df_holdout ensures zero overlap with the FAISS knowledge base.

TARGET_INTENTS = [
    "track_order",
    "get_refund",
    "cancel_order",
    "payment_issue",
    "contact_human_agent",
    "complaint"
]

MULTI_STEP_TEMPLATES = [
    "{orig} Also, please process a refund for order ORD001.",
    "{orig} And cancel order ORD002 as well.",
    "{orig} Additionally, I want to speak to a human agent about order ORD003.",
    "{orig} Also log a complaint about order ORD001.",
    "{orig} And check the payment status for order ORD002 too."
]

if os.path.exists(BENCHMARK_CSV):
    print(f"{BENCHMARK_CSV} already exists - skipping creation.")
    df_bench = pd.read_csv(BENCHMARK_CSV)
else:
    random.seed(RANDOM_SEED)
    frames = []
    for intent in TARGET_INTENTS:
        subset = df_holdout[df_holdout["intent"] == intent].copy()
        n = min(10, len(subset))
        frames.append(subset.sample(n, random_state=RANDOM_SEED))

    df_bench = pd.concat(frames, ignore_index=True)
    df_bench = df_bench[["instruction", "response", "intent", "category"]].copy()
    df_bench.rename(columns={"response": "ground_truth"}, inplace=True)
    df_bench.insert(0, "id", range(1, len(df_bench) + 1))
    df_bench["is_multi_step"] = False

    # Mark natural multi-step rows
    multi_keywords = [" and ", "also", "as well", "plus", "both"]
    multi_mask  = df_bench["instruction"].str.lower().str.contains("|".join(multi_keywords))
    multi_indices = df_bench[multi_mask].index[:10].tolist()

    # Rephrase single-step rows if needed, with real order IDs embedded
    extra_needed = 10 - len(multi_indices)
    if extra_needed > 0:
        candidates = df_bench[~df_bench.index.isin(multi_indices)].sample(
            extra_needed, random_state=99
        )
        for i, (idx, row) in enumerate(candidates.iterrows()):
            template = MULTI_STEP_TEMPLATES[i % len(MULTI_STEP_TEMPLATES)]
            df_bench.at[idx, "instruction"] = template.format(orig=row["instruction"])
            multi_indices.append(idx)

    df_bench.loc[multi_indices, "is_multi_step"] = True
    df_bench.to_csv(BENCHMARK_CSV, index=False)
    print(f"Saved {len(df_bench)} benchmark queries to {BENCHMARK_CSV}")
    print(f"Multi-step queries : {df_bench['is_multi_step'].sum()}")

print(f"\nIntent distribution:")
print(df_bench["intent"].value_counts().to_string())

Saved 60 benchmark queries to benchmark_queries.csv
Multi-step queries : 10

Intent distribution:
intent
track_order            10
get_refund             10
cancel_order           10
payment_issue          10
contact_human_agent    10
complaint              10


In [12]:
# STEP 4 - Embed Training Split and Build FAISS Index

import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print(f"Loading embedding model: {EMBED_MODEL} (CPU)...")
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")

# Knowledge base is ONLY the training split
documents   = df_train["response"].tolist()
doc_intents = df_train["intent"].tolist()
print(f"Knowledge base size: {len(documents):,} documents (training split only)")

print(f"Embedding in batches of {BATCH_SIZE}...")
all_embeddings = []
for start in tqdm(range(0, len(documents), BATCH_SIZE), desc="Embedding"):
    batch = documents[start : start + BATCH_SIZE]
    emb   = embedder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    all_embeddings.append(emb)

embeddings = np.vstack(all_embeddings).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal:,}")

Loading embedding model: all-MiniLM-L6-v2 (CPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base size: 21,497 documents (training split only)
Embedding in batches of 512...


Embedding: 100%|██████████| 42/42 [15:54<00:00, 22.71s/it]

Embeddings shape: (21497, 384)
FAISS index built. Total vectors: 21,497


In [13]:
# STEP 5 - RAG Query Function
#
# Change from previous version:
# The prompt now requires the model to open with an explicit action statement
# in its FIRST sentence before giving the empathetic body response.
# This directly fixes the intent accuracy problem: the LLM judge was seeing
# vague empathetic openers ("I understand your frustration...") and ruling
# that the response did not address the intent.
# The body text remains Bitext-style, preserving BLEU/ROUGE scores.

from groq import Groq

groq_client = Groq(api_key=api_key)

def call_llm(prompt: str, max_retries: int = 3, max_tokens: int = 512,
             temperature: float = 0.3) -> str:
    for attempt in range(1, max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"   Groq error (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(wait)
            else:
                return f"[ERROR: {e}]"

def rag_query(query: str) -> dict:
    """
    Full RAG pipeline:
    embed query -> FAISS search -> prompt Groq LLM

    Prompt change: model must open with an explicit action/information
    statement before the empathetic body. This raises intent accuracy
    without harming BLEU/ROUGE because the body text stays natural.
    """
    t0 = time.time()

    # 1. Embed query
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")

    # 2. Retrieve top-K
    distances, indices    = index.search(q_emb, TOP_K)
    retrieved_docs        = [documents[i] for i in indices[0]]
    retrieved_intents     = [doc_intents[i] for i in indices[0]]

    # 3. Build prompt with explicit action-first instruction
    chunks_text = "\n\n".join([f"Chunk {i+1}: {doc}" for i, doc in enumerate(retrieved_docs)])
    prompt = f"""You are a helpful customer support assistant.
Using ONLY the context below, answer the customer query.

Rules:
1. Your FIRST sentence must explicitly state what action you are taking or
   what information you are providing. Be direct and specific.
   Good examples:
     "I will help you cancel your order."
     "Here is the current status of your order."
     "I can assist you with your refund request."
     "Let me connect you with a human support agent."
   Bad examples (too vague, do NOT use):
     "I understand your frustration."
     "Thank you for contacting us."
2. After the first sentence, give the full helpful response using the context.
3. Do not add information not present in the context.

Context:
{chunks_text}

Customer Query: {query}

Answer:"""

    # 4. Generate response
    response = call_llm(prompt)
    latency  = round(time.time() - t0, 3)

    return {
        "bot_response"      : response,
        "latency_seconds"   : latency,
        "retrieved_intents" : "|".join(retrieved_intents),
        "retrieved_texts"   : " ||| ".join(retrieved_docs)
    }

# Smoke test
print("Smoke test RAG pipeline...")
test = rag_query("Where is my order?")
print(f"   Response  : {test['bot_response'][:180]}...")
print(f"   Latency   : {test['latency_seconds']}s")
print(f"   Retrieved : {test['retrieved_intents']}")


Smoke test RAG pipeline...
   Response  : I will help you locate the status and current location of your order.

To assist you, could you please provide me with the name or tracking number associated with your order? This ...
   Latency   : 0.348s
   Retrieved : track_order|track_order|track_order


In [14]:
# STEP 6 - Run RAG on Benchmark Queries

df_bench = pd.read_csv(BENCHMARK_CSV)

completed_ids = set()
if os.path.exists(RAG_RESULTS_CSV):
    df_existing   = pd.read_csv(RAG_RESULTS_CSV)
    completed_ids = set(df_existing["id"].tolist())
    print(f"Resuming: {len(completed_ids)} queries already done.")
else:
    header_df = pd.DataFrame(columns=[
        "id", "instruction", "ground_truth", "bot_response",
        "intent", "is_multi_step", "latency_seconds",
        "retrieved_intents", "retrieved_texts"
    ])
    header_df.to_csv(RAG_RESULTS_CSV, index=False)
    print(f"Created new {RAG_RESULTS_CSV}")

remaining = df_bench[~df_bench["id"].isin(completed_ids)]
print(f"\nRunning RAG on {len(remaining)} queries...\n")

for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc="RAG queries"):
    result = rag_query(row["instruction"])

    record = pd.DataFrame([{
        "id"               : row["id"],
        "instruction"      : row["instruction"],
        "ground_truth"     : row["ground_truth"],
        "bot_response"     : result["bot_response"],
        "intent"           : row["intent"],
        "is_multi_step"    : row["is_multi_step"],
        "latency_seconds"  : result["latency_seconds"],
        "retrieved_intents": result["retrieved_intents"],
        "retrieved_texts"  : result["retrieved_texts"]
    }])
    record.to_csv(RAG_RESULTS_CSV, mode="a", header=False, index=False)
    time.sleep(0.5)

print(f"\nResults saved to {RAG_RESULTS_CSV}")
df_results = pd.read_csv(RAG_RESULTS_CSV)
print(f"   Total rows  : {len(df_results)}")
print(f"   Avg latency : {df_results['latency_seconds'].mean():.2f}s")


Created new rag_results.csv

Running RAG on 60 queries...



RAG queries:  77%|███████▋  | 46/60 [04:27<01:41,  7.23s/it]

   Groq error (attempt 1/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01knyhk036e8ybegp6q7kv14zf` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5119, Requested 912. Please try again in 310ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


RAG queries: 100%|██████████| 60/60 [05:59<00:00,  6.00s/it]


Results saved to rag_results.csv
   Total rows  : 60
   Avg latency : 5.49s


In [15]:
# STEP 7 - BLEU + ROUGE Scores for RAG
#
# Change from previous version:
# normalize_for_nlg() strips generated ticket/order IDs and amounts before
# scoring. These tokens never appear in the Bitext ground truth, so they
# kill BLEU precision. Normalizing them to placeholders gives a fairer
# lexical comparison while preserving all meaningful content words.

from rouge_score import rouge_scorer as rouge_scorer_lib
import sacrebleu
import re

rouge = rouge_scorer_lib.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def normalize_for_nlg(text: str) -> str:
    """
    Replace generated IDs and amounts with neutral placeholders before
    BLEU/ROUGE scoring. These tokens are correct and useful in the response
    but have zero chance of appearing in the Bitext reference text,
    so they unfairly penalize lexical overlap metrics.
    """
    text = re.sub(r"\b(REF|CAN|AGT|CMP|TKT)\d{4,6}\b", "TICKET_ID", text)
    text = re.sub(r"\bORD\d{3}\b", "ORDER_ID", text)
    text = re.sub(r"Rs\.\s*[\d,]+", "AMOUNT", text)
    return text

def compute_nlg_metrics(prediction: str, reference: str) -> dict:
    pred_norm = normalize_for_nlg(prediction)
    r = rouge.score(reference, pred_norm)
    b = sacrebleu.sentence_bleu(pred_norm, [reference])
    return {
        "rouge1" : round(r["rouge1"].fmeasure, 3),
        "rougeL" : round(r["rougeL"].fmeasure, 3),
        "bleu"   : round(b.score, 2)
    }

df_results = pd.read_csv(RAG_RESULTS_CSV)
nlg_rows   = [compute_nlg_metrics(row["bot_response"], row["ground_truth"])
              for _, row in df_results.iterrows()]
df_nlg = pd.DataFrame(nlg_rows)

print("\nRAG NLG Metrics (vs held-out ground truth, IDs normalized):")
print(f"   BLEU   : {df_nlg['bleu'].mean():.2f}")
print(f"   ROUGE-1: {df_nlg['rouge1'].mean():.3f}")
print(f"   ROUGE-L: {df_nlg['rougeL'].mean():.3f}")

df_results = pd.concat([df_results, df_nlg], axis=1)
df_results.to_csv(RAG_RESULTS_CSV, index=False)
print(f"NLG scores saved back to {RAG_RESULTS_CSV}")



RAG NLG Metrics (vs held-out ground truth, IDs normalized):
   BLEU   : 12.91
   ROUGE-1: 0.487
   ROUGE-L: 0.304
NLG scores saved back to rag_results.csv


In [16]:
# STEP 8 - Save to Google Drive

from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copy(BENCHMARK_CSV,   '/content/drive/MyDrive/benchmark_queries.csv')
shutil.copy(RAG_RESULTS_CSV, '/content/drive/MyDrive/rag_results.csv')
print("Both files saved to Google Drive.")

Mounted at /content/drive
Both files saved to Google Drive.


In [17]:
# STEP 9 - Gradio Chat UI

import gradio as gr

def chat_rag(message, history):
    result       = rag_query(message)
    footer       = (f"\n\n---\nLatency: {result['latency_seconds']}s"
                    f" | Retrieved intents: {result['retrieved_intents']}")
    return result["bot_response"] + footer

demo = gr.ChatInterface(
    fn=chat_rag,
    title="RAG Customer Support Bot",
    description=(
        "Powered by FAISS + Sentence Transformers + Groq Llama 3.1.\n"
        "Try: 'Where is my order?' or 'How do I get a refund?'"
    ),
    examples=[
        "Where is my order?",
        "I want to cancel my order.",
        "How do I get a refund?",
        "My payment failed, what do I do?",
        "I want to speak to a human agent."
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://952ac549a6597d7dcb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
